# Introduction to the HTRC Extracted Features API

Created by Janet Swatscheno under [Creative Commons CC BY License](https://creativecommons.org/licenses/by/4.0/).

## Helpful Links
- [Extracted Features API Documentation](https://htrc.stoplight.io/docs/ef-api/8xpvh96ani2e0-ef-api)
- [HathiTrust Digital Library](https://www.hathitrust.org/)

## TORCHLITE

The [TORCHLITE project](https://ischool.illinois.edu/news-events/news/2022/01/hathitrust-research-center-receives-neh-support-open-research-tools) extends volume-level, page-level, and fine-grain access to our [Extracted Features (EF) dataset](https://analytics.hathitrust.org/deriveddatasets), which contains metadata and statistical information extracted from the full-text data representing 17.5 million digitized volumes in the HathiTrust Digital Library. As part of TORCHLITE, HTRC has developed a well-documented API to allow our user community to develop its own tools for interacting with data. We are also developing a web-based, interactive visualization dashboard using the API.

The following notebook demonstrated the different API calls you can use to access extracted features data.
____

## Setup
### ➡️ Importing
The API returns JSON files. We'll import the JSON library, which will make reading the responses from the API easier.

In [ ]:
import json

### 🛀 Cleaning the HathiTrust id

*Note*: Not all of the HathiTrust ids are formatted in a way we can easily use because they include special characters. For example, some HathiTrust IDs use "ark" ids which include forward slashes and colons which can't be passed through the API. We can swap out those characters to pass them through the API.

Example:

**uc2.ark:/13960/t0zp3zt0v**

becomes

**uc2.ark+=13960=t0zp3zt0v**

Below, we define a function that will clean the id for us by replacing the characters we can't use. We'll use this in the future for passing ids to the API.

*Note: We only need "clean ids" when we're using the volume id to make a call to the API. For worksets, we don't need to work with "clean ids"*

In [ ]:
def clean_id(id: str):
    return id.replace(':', '+').replace('/', '=')

In [ ]:
ht_id = "uc2.ark:/13960/t0zp3zt0v"
clean_htid = clean_id(ht_id)
print(clean_htid)

uc2.ark+=13960=t0zp3zt0v


## ☎️ TORCHLITE API calls for individual volumes

The following API calls get data and/or metadata for individual volumes from the HathiTrust corpus. For working with multiple volumes, you can create worksets which will return data for all the volumes in a workset.

### 📕 Get Extracted Features data for a volume by volume ID

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

headers = { 'Accept': "application/json" }

conn.request("GET", f"/ef-api/volumes/{clean_htid}", headers=headers)

res = conn.getresponse()
data = res.read()

# Decode the response and load as JSON
parsed_data = json.loads(data.decode("utf-8"))

# Pretty-print the JSON data. Limited to 1,000 lines so it will print.
print(json.dumps(parsed_data, indent=4)[:1000])

{
    "code": 200,
    "data": {
        "@context": "https://worksets.htrc.illinois.edu/context/ef_context.jsonld",
        "schemaVersion": "https://schemas.hathitrust.org/EF_Schema_v_3.0",
        "id": "https://data.analytics.hathitrust.org/extracted-features/20200210/uc2.ark:/13960/t0zp3zt0v",
        "htid": "uc2.ark:/13960/t0zp3zt0v",
        "type": "DataFeed",
        "publisher": {
            "id": "https://analytics.hathitrust.org",
            "type": "Organization",
            "name": "HathiTrust Research Center"
        },
        "datePublished": 20200210,
        "metadata": {
            "schemaVersion": "https://schemas.hathitrust.org/EF_Schema_MetadataSubSchema_v_3.0",
            "id": "http://hdl.handle.net/2027/uc2.ark:/13960/t0zp3zt0v",
            "type": [
                "DataFeedItem",
                "Book"
            ],
            "dateCreated": 20200209,
            "title": "Jane Eyre /",
            "contributor": {
                "id": "http://www.

### 🔍 Check if a volume exists using a volume ID

We might want to just check if a HathiTrust item exists as an extracted features file, without getting all of the data itself. We can do that here by passing a clean HTID and printing the status code

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

conn.request("HEAD", f"/ef-api/volumes/{clean_htid}")

res = conn.getresponse()
data = res.read()

print(f"{res.status} {res.reason}")

200 OK


#### ⛏️ Let's dig into the response we got from the API

In our API response, there are two possible responses:

- code: 200: This indicates that the HTTP status code of the response is "200 OK," which means that the request was successful and the server is returning data. (yay!)
- code: 404 Not found means that data is not available for that particular HathiTrust item

### 📗 Get volume metadata by volume id

For some applications, you might only need metadata for a volume, rather than all of the features. You can use the Get volume metadata by volume id call to return only metadata for an item. Again, we're going to use a "clean id" to pass to the API and return the metadata.

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

headers = { 'Accept': "application/json" }

conn.request("GET", f"/ef-api/volumes/{clean_htid}/metadata", headers=headers)

res = conn.getresponse()
data = res.read()

# Decode the response and load as JSON
parsed_data = json.loads(data.decode("utf-8"))

# Pretty-print the JSON data. Limited to 10,000 lines.
print(json.dumps(parsed_data, indent=4)[:10000])

{
    "code": 200,
    "data": {
        "htid": "uc2.ark:/13960/t0zp3zt0v",
        "metadata": {
            "schemaVersion": "https://schemas.hathitrust.org/EF_Schema_MetadataSubSchema_v_3.0",
            "id": "http://hdl.handle.net/2027/uc2.ark:/13960/t0zp3zt0v",
            "type": [
                "DataFeedItem",
                "Book"
            ],
            "dateCreated": 20200209,
            "title": "Jane Eyre /",
            "contributor": {
                "id": "http://www.viaf.org/viaf/71388025",
                "type": "http://id.loc.gov/ontologies/bibframe/Person",
                "name": "Bronte\u0308, Charlotte, 1816-1855."
            },
            "pubDate": 1899,
            "publisher": {
                "id": "http://catalogdata.library.illinois.edu/lod/entities/ProvisionActivityAgent/ht/Harper%20&%20brothers",
                "type": "http://id.loc.gov/ontologies/bibframe/Organization",
                "name": "Harper & brothers"
            },
          

### 📃 Get subset of pages of volume by volume id

It is also possible that you're not interested in using all of the pages of a volume. Perhaps you are only interested in returning specific pages from a volume.

For this API, you need to privde two things
- The id of the volume
- The sequence formatted in a very specific way (00000004%2C00000005%2C00000006)

Below is a function that takes a start page and end page and formats them correctly for the API.

In [ ]:
def transform_page_range(start, end):
    # Generate a list of page numbers within the range
    page_numbers = [f"{i:08d}" for i in range(start, end+1)]
    # Join the list into a single string with '%2C' as the separator
    return "%2C".join(page_numbers)

# Enter a start page and an end page
start_page = 4
end_page = 11
pages = transform_page_range(start_page, end_page)
print(pages)

00000004%2C00000005%2C00000006%2C00000007%2C00000008%2C00000009%2C00000010%2C00000011


Next, we'll pass the clean id and the page sequence to the API call as "pages".

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

headers = { 'Accept': "application/json" }

conn.request("GET", f"/ef-api/volumes/{clean_htid}/pages?seq={pages}", headers=headers)

res = conn.getresponse()
data = res.read()

# Decode the response and load as JSON
parsed_data = json.loads(data.decode("utf-8"))

# Pretty-print the JSON data. Limited to 1,000 lines.
print(json.dumps(parsed_data, indent=4)[:1000])

{
    "code": 200,
    "data": {
        "htid": "uc2.ark:/13960/t0zp3zt0v",
        "pages": [
            {
                "seq": "00000004",
                "version": "d41d8cd98f00b204e9800998ecf8427e",
                "tokenCount": 0,
                "lineCount": 0,
                "emptyLineCount": 0,
                "sentenceCount": 0,
                "header": null,
                "body": null,
                "footer": null,
                "calculatedLanguage": null
            },
            {
                "seq": "00000005",
                "version": "d24c4c4f012a083f368244592c16a98f",
                "tokenCount": 43,
                "lineCount": 27,
                "emptyLineCount": 13,
                "sentenceCount": 1,
                "header": null,
                "body": {
                    "tokenCount": 43,
                    "lineCount": 27,
                    "emptyLineCount": 13,
                    "sentenceCount": 1,
                    "capAlphaSeq":

## ☎️ TORCHLITE API calls for worksets (multiple volumes)
The following API calls use the concept of worksets, or a collection of multiple HathiTrust volumes, to get data from the API. Before you can get data from the API using a workset, you have to use the API to create the workset.

### 🔨 Create workset


In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

payload = "uc1.32106011187561 mdp.35112103187797 uc1.$b684263 mdp.39015058744122"

headers = {
    'Content-Type': "text/plain",
    'Accept': "application/json"
}

conn.request("POST", "/ef-api/worksets", payload, headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"code":200,"data":{"id":"671918b63800004901c7166a","htids":["mdp.35112103187797","mdp.39015058744122","uc1.$b684263","uc1.32106011187561"],"created":"2024-10-23T15:39:34.909Z"}}


### 🗑️ Delete workset by workset id

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

headers = { 'Accept': "application/json" }

conn.request("DELETE", "/ef-api/worksets/671033ad3b000011012ce421", headers=headers) // change the id to the workset you want to delete

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

SyntaxError: invalid syntax (1662726284.py, line 7)

### 📚 Get workset by workset id

Retrieves the HathiTrust ids of the volumes in a workset and the date and time a workset was created.

In [ ]:
workset = "66477e0b2600006a07132b28"

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

headers = { 'Accept': "application/json" }

conn.request("GET", f"/ef-api/worksets/{workset}", headers=headers)

res = conn.getresponse()
data = res.read()

print(data.decode("utf-8"))

{"code":200,"data":{"id":"66477e0b2600006a07132b28","htids":["uc1.32106008055367","uc1.32106007171835","uc1.32106009338549","uc1.32106009087138","mdp.49015003071447","uc1.32106007983098"],"created":"2024-05-17T15:55:55.893Z"}}


### 📚 Get workset volumes by workset id

Returns volume metadata and feature data for all of the volumes in a workset. This call returns a lot of data, so it can take awhile to load. It returns feature data at the page level, meaning it returns every word on a page in a "bag of words" format. A faster alternative to this call is using the Get workset volumes by workset id (aggregated) which will return aggregated data for the volume.

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

headers = { 'Accept': "application/json" }

conn.request("GET", f"/ef-api/worksets/{workset}/volumes/aggregated", headers=headers)

res = conn.getresponse()
data = res.read()

# Decode the response and load as JSON
parsed_data = json.loads(data.decode("utf-8"))

# Pretty-print the JSON data. Limited to 1,000 lines.
print(json.dumps(parsed_data, indent=4)[:1000])

{
    "code": 200,
    "data": [
        {
            "htid": "mdp.49015003071447",
            "metadata": {
                "schemaVersion": "https://schemas.hathitrust.org/EF_Schema_MetadataSubSchema_v_3.0",
                "id": "http://hdl.handle.net/2027/mdp.49015003071447",
                "type": [
                    "DataFeedItem",
                    "Book"
                ],
                "dateCreated": 20200209,
                "title": "Dune /",
                "contributor": {
                    "id": "http://www.viaf.org/viaf/59083797",
                    "type": "http://id.loc.gov/ontologies/bibframe/Person",
                    "name": "Herbert, Frank."
                },
                "pubDate": 1965,
                "publisher": {
                    "id": "http://catalogdata.library.illinois.edu/lod/entities/ProvisionActivityAgent/ht/Ace%20Books",
                    "type": "http://id.loc.gov/ontologies/bibframe/Organization",
                    "name": "A

### 📚 Get workset volumes(aggregated) by workset id
Returns volume metadata and aggregated feature data for all of the volumes in a workset. It returns feature data at the volume level, meaning it returns every word in a volume in a "bag of words" format.

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

headers = { 'Accept': "application/json" }

conn.request("GET", f"/ef-api/worksets/{workset}/volumes/aggregated", headers=headers)

res = conn.getresponse()
data = res.read()

# Decode the response and load as JSON
parsed_data = json.loads(data.decode("utf-8"))

# Pretty-print the JSON data Limited to 1,000 lines.
print(json.dumps(parsed_data, indent=4)[:1000])

{
    "code": 200,
    "data": [
        {
            "htid": "mdp.49015003071447",
            "metadata": {
                "schemaVersion": "https://schemas.hathitrust.org/EF_Schema_MetadataSubSchema_v_3.0",
                "id": "http://hdl.handle.net/2027/mdp.49015003071447",
                "type": [
                    "DataFeedItem",
                    "Book"
                ],
                "dateCreated": 20200209,
                "title": "Dune /",
                "contributor": {
                    "id": "http://www.viaf.org/viaf/59083797",
                    "type": "http://id.loc.gov/ontologies/bibframe/Person",
                    "name": "Herbert, Frank."
                },
                "pubDate": 1965,
                "publisher": {
                    "id": "http://catalogdata.library.illinois.edu/lod/entities/ProvisionActivityAgent/ht/Ace%20Books",
                    "type": "http://id.loc.gov/ontologies/bibframe/Organization",
                    "name": "A

### 📚 Get workset volumes metadata by workset id

In [ ]:
import http.client

conn = http.client.HTTPSConnection("data.htrc.illinois.edu")

conn.request("GET", f"/ef-api/worksets/{workset}/metadata")

res = conn.getresponse()
data = res.read()

# Decode the response and load as JSON
parsed_data = json.loads(data.decode("utf-8"))

# Pretty-print the JSON data.Limited to 1,000 lines.
print(json.dumps(parsed_data, indent=4)[:1000])

{
    "code": 200,
    "data": [
        {
            "htid": "mdp.49015003071447",
            "metadata": {
                "schemaVersion": "https://schemas.hathitrust.org/EF_Schema_MetadataSubSchema_v_3.0",
                "id": "http://hdl.handle.net/2027/mdp.49015003071447",
                "type": [
                    "DataFeedItem",
                    "Book"
                ],
                "dateCreated": 20200209,
                "title": "Dune /",
                "contributor": {
                    "id": "http://www.viaf.org/viaf/59083797",
                    "type": "http://id.loc.gov/ontologies/bibframe/Person",
                    "name": "Herbert, Frank."
                },
                "pubDate": 1965,
                "publisher": {
                    "id": "http://catalogdata.library.illinois.edu/lod/entities/ProvisionActivityAgent/ht/Ace%20Books",
                    "type": "http://id.loc.gov/ontologies/bibframe/Organization",
                    "name": "A